  ## KURAMOTO OSCILLATORS ON WATTS-STROGATZ SMALL-WORLD NETWORKS
  
  System:  dθᵢ/dt = ωᵢ  +  (K/N) Σⱼ aᵢⱼ sin(θⱼ − θᵢ)
 
  where
    θᵢ  – phase of oscillator i

    ωᵢ  – natural frequency of oscillator i

    K   – global coupling strength

    aᵢⱼ – adjacency matrix of the WS small-world graph

    N   – number of oscillators
 
  The base ring lattice has degree 6 (each node connected to 3 nearest
  neighbours on each side), then edges are rewired with probability p.


In [1]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib.animation import FuncAnimation
from scipy.integrate import solve_ivp
from scipy.signal import welch
from mpl_toolkits.axes_grid1 import make_axes_locatable
import warnings
warnings.filterwarnings("ignore")

In [4]:
# Defining global color scheme and random seed for reproducibility

np.random.seed(42)
plt.style.use("dark_background")
CMAP_PHASE  = "hsv"          # circular colormap for phases
CMAP_ORDER  = "plasma"
CMAP_HEAT   = "inferno"
ACCENT1     = "#00FFCC"      # cyan-green
ACCENT2     = "#FF4F8B"      # hot pink
ACCENT3     = "#FFD700"      # gold
BG          = "#0A0A0F"

In [5]:
# NETWORK CONSTRUCTION

def make_ws_network(N=100, k=6, p=0.1):
    """
    Build a Watts–Strogatz small-world graph.

    N : number of nodes
    k : each node is connected to k nearest neighbours on the ring (must be even)
    p : rewiring probability  (p=0 → regular ring; p=1 → random graph)

    G : networkx Graph
    A : NxN numpy adjacency matrix"""

    G = nx.watts_strogatz_graph(N, k, p, seed=42)
    A = nx.to_numpy_array(G)          # float64 adjacency matrix
    return G, A
 
 
def network_stats(G):
    """Compute and return a dict of key network measures."""
    stats = {}
    stats["N"]          = G.number_of_nodes()
    stats["E"]          = G.number_of_edges()
    stats["avg_degree"] = np.mean([d for _, d in G.degree()])
    stats["clustering"] = nx.average_clustering(G)
 
    # mean path length (only on largest connected component to avoid infinite distances)
    lcc = max(nx.connected_components(G), key=len)
    Gsub = G.subgraph(lcc)
    stats["mean_path"]  = nx.average_shortest_path_length(Gsub)
    return stats

In [6]:
# KURAMOTO ODE
 
def kuramoto_rhs(t, theta, omega, K, A, N):
    # Compute the phase difference matrix:
    delta = theta[np.newaxis, :] - theta[:, np.newaxis]   # shape (N, N)
 
    # Weighted sum of sin(delta θ) over neighbours
    coupling = (K / N) * np.sum(A * np.sin(delta), axis=1)
 
    return omega + coupling
 
 
def simulate(N=100, k=6, p=0.1, K=2.0,
             t_end=50.0, dt=0.05, freq_std=1.0):
    """
    t_end    : total simulation time
    dt       : time step
    freq_std : std of natural frequencies (Gaussian distribution)
    """
    G, A = make_ws_network(N, k, p)
 
    # Natural frequencies drawn from a zero-mean Gaussian
    omega = np.random.normal(0.0, freq_std, N)
 
    # Random initial phases uniformly in [0, 2π)
    theta0 = np.random.uniform(0, 2 * np.pi, N)
 
    t_eval = np.arange(0, t_end, dt)
 
    sol = solve_ivp(
        fun=kuramoto_rhs,
        t_span=(0, t_end),
        y0=theta0,
        t_eval=t_eval,
        args=(omega, K, A, N),
        method="RK45",
        rtol=1e-6,
        atol=1e-8,
    )
    return sol.t, sol.y.T, omega, G, A   # theta shape: (time, N)

In [7]:
# ORDER PARAMETER & DIAGNOSTICS
 
def order_parameter(theta):
    """
    Compute the Kuramoto order parameter r(t) and mean phase Ψ(t).
    r(t) e^{iΨ(t)} = (1/N) Σⱼ e^{iθⱼ(t)} 
    r=1  → perfect synchrony
    r=0  → incoherence
    """
    z   = np.mean(np.exp(1j * theta), axis=1)   # shape: (time,)
    r   = np.abs(z)
    psi = np.angle(z)
    return r, psi
 
 
def time_averaged_r(theta, frac=0.5):
    """
    Return the time-averaged order parameter over the last "frac" of the run
    (discards transient).
    """
    r, _ = order_parameter(theta)
    start = int(len(r) * (1 - frac))
    return np.mean(r[start:])
 
 
def phase_coherence_matrix(theta_snapshot):
    """
    Pairwise phase coherence between oscillators at a given time snapshot.
    for a single snapshot this is just cos( delta θ).
    """
    N = len(theta_snapshot)
    diff = theta_snapshot[:, np.newaxis] - theta_snapshot[np.newaxis, :]  # (N,N)
    return np.abs(np.cos(diff))    # symmetric, values in [0,1]
 
 
def local_order_parameter(theta, A):
    """
    Local (node-level) order parameter: synchrony of each node's neighbours.
    """
    N   = theta.shape[1]
    deg = A.sum(axis=1)            # degree of each node
    # For each time step compute local r
    exp_theta = np.exp(1j * theta)                   # (T, N)
    local_z   = exp_theta @ A.T / np.maximum(deg, 1) # (T, N)
    return np.abs(local_z)                            # (T, N)

In [8]:
# K SWEEP STUDIES
 
def sweep_K(N=100, k=6, p=0.1, K_values=None,
            t_end=60, dt=0.05, freq_std=1.0, n_trials=3):
    """
    Sweep coupling strength K and compute steady-state order parameter.
    Averages over n_trials different network/frequency realisations.
    """
    if K_values is None:
        K_values = np.linspace(0, 6, 30)
 
    r_mean = np.zeros(len(K_values))
    r_std  = np.zeros(len(K_values))
 
    for idx, K in enumerate(K_values):
        rs = []
        for trial in range(n_trials):
            np.random.seed(trial * 100 + idx)
            G, A   = make_ws_network(N, k, p)
            omega  = np.random.normal(0, freq_std, N)
            theta0 = np.random.uniform(0, 2 * np.pi, N)
            t_eval = np.arange(0, t_end, dt)
            sol    = solve_ivp(kuramoto_rhs, (0, t_end), theta0, t_eval=t_eval,
                               args=(omega, K, A, N), method="RK45",
                               rtol=1e-6, atol=1e-8)
            theta  = sol.y.T
            rs.append(time_averaged_r(theta))
        r_mean[idx] = np.mean(rs)
        r_std[idx]  = np.std(rs)
        print(f"  K={K:.2f}  r={r_mean[idx]:.3f} ± {r_std[idx]:.3f}")
 
    return K_values, r_mean, r_std
 
 
def sweep_p(N=100, k=6, p_values=None, K=3.0,
            t_end=60, dt=0.05, freq_std=1.0, n_trials=3):
    """
    Sweep rewiring probability p and compute steady-state order parameter.
    """
    if p_values is None:
        p_values = np.logspace(-3, 0, 20)
 
    r_mean = np.zeros(len(p_values))
    r_std  = np.zeros(len(p_values))
 
    for idx, p in enumerate(p_values):
        rs = []
        for trial in range(n_trials):
            np.random.seed(trial * 100 + idx)
            G, A   = make_ws_network(N, k, p)
            omega  = np.random.normal(0, freq_std, N)
            theta0 = np.random.uniform(0, 2 * np.pi, N)
            t_eval = np.arange(0, t_end, dt)
            sol    = solve_ivp(kuramoto_rhs, (0, t_end), theta0, t_eval=t_eval,
                               args=(omega, K, A, N), method="RK45",
                               rtol=1e-6, atol=1e-8)
            theta  = sol.y.T
            rs.append(time_averaged_r(theta))
        r_mean[idx] = np.mean(rs)
        r_std[idx]  = np.std(rs)
        print(f"  p={p:.4f}  r={r_mean[idx]:.3f} ± {r_std[idx]:.3f}")
 
    return p_values, r_mean, r_std
 
 
def phase_diagram(N=80, k=6,
                  K_vals=None, p_vals=None,
                  t_end=50, dt=0.05, freq_std=1.0):
    """
    2-D phase diagram in (p, K) space. Returns r matrix.
    """
    if K_vals is None:
        K_vals = np.linspace(0, 6, 15)
    if p_vals is None:
        p_vals = np.logspace(-3, 0, 12)
 
    R = np.zeros((len(K_vals), len(p_vals)))
 
    for ki, K in enumerate(K_vals):
        for pi, p in enumerate(p_vals):
            np.random.seed(ki * 100 + pi)
            G, A   = make_ws_network(N, k, p)
            omega  = np.random.normal(0, freq_std, N)
            theta0 = np.random.uniform(0, 2 * np.pi, N)
            t_eval = np.arange(0, t_end, dt)
            sol    = solve_ivp(kuramoto_rhs, (0, t_end), theta0, t_eval=t_eval,
                               args=(omega, K, A, N), method="RK45",
                               rtol=1e-6, atol=1e-8)
            R[ki, pi] = time_averaged_r(sol.y.T)
        print(f"  K={K:.2f} done")
 
    return K_vals, p_vals, R

In [ ]:
# PLOTTING FUNCTIONS
 
# Network visualisation
 
def plot_networks(p_values=(0.0, 0.05, 0.5, 1.0), N=50, k=6):
    """
    Show the WS network at four rewiring probabilities, with node colour
    showing degree.
    """
    fig, axes = plt.subplots(1, 4, figsize=(20, 5),
                             facecolor=BG)
    fig.suptitle("Watts–Strogatz Networks  (N=50, k=6)",
                 color="white", fontsize=14, fontweight="bold", y=1.02)
 
    for ax, p in zip(axes, p_values):
        G, A = make_ws_network(N, k, p)
        pos  = nx.circular_layout(G)
 
        degrees = np.array([d for _, d in G.degree()])
        nx.draw_networkx_edges(G, pos, ax=ax,
                               alpha=0.3, edge_color="#444466", width=0.7)
        nx.draw_networkx_nodes(G, pos, ax=ax,
                               node_size=60,
                               node_color=degrees,
                               cmap="plasma",
                               vmin=degrees.min(), vmax=degrees.max())
        ax.set_facecolor(BG)
        ax.set_title(f"p = {p}", color="white", fontsize=11)
        ax.axis("off")
 
        # small stats annotation
        stats = network_stats(G)
        info  = (f"C={stats['clustering']:.3f}\n"
                 f"L={stats['mean_path']:.2f}")
        ax.text(0.02, 0.02, info, transform=ax.transAxes,
                color=ACCENT1, fontsize=8, va="bottom", family="monospace")
 
    plt.tight_layout()
    plt.savefig("fig1_ws_networks.png",
                dpi=150, bbox_inches="tight", facecolor=BG)
    plt.show()
 
 
# Order parameter vs time
 
def plot_order_vs_time(N=100, k=6, p=0.1,
                       K_vals=(0.5, 1.5, 3.0, 5.0)):
    """
    Show r(t) for several coupling strengths on the same WS network.
    Also shows the raw phase trajectories for the strongest coupling case.
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor=BG)
    ax_r, ax_theta = axes
 
    G, A = make_ws_network(N, k, p)
 
    colors = plt.cm.plasma(np.linspace(0.15, 0.95, len(K_vals)))
 
    for K, col in zip(K_vals, colors):
        np.random.seed(7)
        omega  = np.random.normal(0, 1.0, N)
        theta0 = np.random.uniform(0, 2 * np.pi, N)
        t_eval = np.arange(0, 60, 0.05)
        sol    = solve_ivp(kuramoto_rhs, (0, 60), theta0, t_eval=t_eval,
                           args=(omega, K, A, N), method="RK45",
                           rtol=1e-6, atol=1e-8)
        theta  = sol.y.T
        r, _   = order_parameter(theta)
        ax_r.plot(sol.t, r, color=col, lw=1.5, label=f"K={K}")
 
        # show phase trajectories only for strongest coupling
        if K == K_vals[-1]:
            for i in range(0, N, 5):    # plot every 5th oscillator
                # wrap phase to [0, 2π] for display
                ax_theta.plot(sol.t, np.mod(theta[:, i], 2*np.pi),
                              color=col, alpha=0.3, lw=0.5)
 
    # decorate order-parameter panel
    ax_r.set_xlabel("Time", color="white")
    ax_r.set_ylabel("Order parameter  r(t)", color="white")
    ax_r.set_title("Synchronisation onset  (p=0.1)", color="white")
    ax_r.legend(framealpha=0.2, labelcolor="white")
    ax_r.set_ylim(0, 1.05)
    ax_r.tick_params(colors="white")
    ax_r.spines[:].set_color("#444")
    ax_r.axhline(1.0, ls="--", color="white", alpha=0.2)
 
    # decorate phase panel
    ax_theta.set_xlabel("Time", color="white")
    ax_theta.set_ylabel("Phase  θᵢ(t)  [mod 2π]", color="white")
    ax_theta.set_title(f"Phase trajectories  (K={K_vals[-1]}, every 5th node)",
                       color="white")
    ax_theta.tick_params(colors="white")
    ax_theta.spines[:].set_color("#444")
 
    plt.tight_layout()
    plt.savefig("fig2_order_vs_time.png",dpi=150, bbox_inches="tight", facecolor=BG)
    plt.show()
 
 
# r vs K sweep (bifurcation diagram)
 
def plot_r_vs_K(K_values, r_mean, r_std, p):
    """
    Classic bifurcation-style plot: steady-state r vs coupling K.
    Marks the analytical Kuramoto critical coupling Kc = 2/πg(0)
    where g(0) = 1/(π·sigma) for a Lorentzian with half-width σ,
    or g(0) = 1/(sigma√(2π)) for Gaussian.
    """
    sigma = 1.0
    Kc_gaussian = 2 * np.sqrt(2 * np.pi) * sigma   # = 2/(g(0)) for Gaussian
    # For the mean-field all-to-all: Kc = 2σ√(2/π) ≈ 1.596σ for Gaussian
    Kc_mf = 2 * sigma * np.sqrt(2 / np.pi)
 
    fig, ax = plt.subplots(figsize=(9, 5), facecolor=BG)
    ax.set_facecolor(BG)
 
    ax.fill_between(K_values, r_mean - r_std, r_mean + r_std,
                    alpha=0.25, color=ACCENT1)
    ax.plot(K_values, r_mean, color=ACCENT1, lw=2, label="⟨r⟩ (WS network)")
 
    # analytical mean-field prediction (all-to-all):
    # r_mf = 0 for K < Kc, r_mf ≈ sqrt(1 - Kc/K) for K > Kc
    K_plot = np.linspace(Kc_mf, K_values[-1], 200)
    r_mf   = np.sqrt(1 - Kc_mf / K_plot)
    ax.plot(K_plot, r_mf, "--", color=ACCENT2, lw=1.5,
            label=f"Mean-field (Kc={Kc_mf:.2f})")
 
    ax.axvline(Kc_mf, ls=":", color=ACCENT3, alpha=0.7,
               label=f"Mean-field Kc = {Kc_mf:.2f}")
 
    ax.set_xlabel("Coupling strength  K", color="white", fontsize=12)
    ax.set_ylabel("Steady-state order parameter  ⟨r⟩", color="white", fontsize=12)
    ax.set_title(f"Bifurcation diagram  (p={p})", color="white", fontsize=13)
    ax.legend(framealpha=0.15, labelcolor="white")
    ax.tick_params(colors="white")
    ax.spines[:].set_color("#444")
    ax.set_ylim(0, 1.05)
 
    plt.tight_layout()
    plt.savefig("fig3_r_vs_K.png",
                dpi=150, bbox_inches="tight", facecolor=BG)
    plt.show()
 
 
# r vs p sweep
 
def plot_r_vs_p(p_values, r_mean, r_std, K):
    """
    How does synchronisation level change with network randomness?
    """
    fig, ax = plt.subplots(figsize=(9, 5), facecolor=BG)
    ax.set_facecolor(BG)
 
    ax.fill_between(p_values, r_mean - r_std, r_mean + r_std,
                    alpha=0.25, color=ACCENT2)
    ax.plot(p_values, r_mean, "o-", color=ACCENT2, lw=2, ms=5,
            label=f"⟨r⟩  (K={K})")
 
    ax.set_xscale("log")
    ax.set_xlabel("Rewiring probability  p", color="white", fontsize=12)
    ax.set_ylabel("Steady-state order parameter  ⟨r⟩", color="white", fontsize=12)
    ax.set_title(f"Synchronisation vs network topology  (K={K})",
                 color="white", fontsize=13)
    ax.legend(framealpha=0.15, labelcolor="white")
    ax.tick_params(colors="white")
    ax.spines[:].set_color("#444")
    ax.set_ylim(0, 1.05)
 
    # Annotate regimes
    ax.axvspan(1e-3, 1e-2, alpha=0.08, color="blue")
    ax.axvspan(1e-2, 1e-1, alpha=0.08, color="green")
    ax.axvspan(1e-1, 1.0,  alpha=0.08, color="red")
    ax.text(2e-3,  0.05, "Regular\nlattice", color="skyblue",  fontsize=8)
    ax.text(2e-2,  0.05, "Small\nworld",     color="limegreen",fontsize=8)
    ax.text(2e-1,  0.05, "Random\ngraph",    color="#FF8888",  fontsize=8)
 
    plt.tight_layout()
    plt.savefig("fig4_r_vs_p.png",
                dpi=150, bbox_inches="tight", facecolor=BG)
    plt.show()
 
 
# Phase snapshot on circular layout
 
def plot_phase_snapshot(theta, G, title="Phase snapshot", fname="fig5_phase_snap.png"):
    """
    Draw the network with nodes coloured by current phase (circular colormap).
    Edge colour shows pairwise phase difference.
    """
    fig, ax = plt.subplots(figsize=(9, 9), facecolor=BG)
    ax.set_facecolor(BG)
 
    pos = nx.circular_layout(G)
    phases = np.mod(theta, 2 * np.pi)      # wrap to [0, 2π]
 
    # draw edges, coloured by |Δθ|
    edges = list(G.edges())
    for u, v in edges:
        diff = np.abs(phases[u] - phases[v])
        diff = min(diff, 2 * np.pi - diff)   # shortest angular distance
        col  = plt.cm.RdYlGn(1 - diff / np.pi)  # green=sync, red=async
        nx.draw_networkx_edges(G, pos, edgelist=[(u, v)], ax=ax,
                               edge_color=[col], alpha=0.5, width=1.0)
 
    # draw nodes with circular colormap
    node_col = [plt.cm.hsv(p / (2 * np.pi)) for p in phases]
    nx.draw_networkx_nodes(G, pos, node_color=node_col, node_size=120, ax=ax)
 
    # colour bar for phases
    sm = plt.cm.ScalarMappable(cmap="hsv",
                                norm=plt.Normalize(vmin=0, vmax=2 * np.pi))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label("Phase θ [rad]", color="white")
    cbar.ax.yaxis.set_tick_params(color="white")
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")
 
    ax.set_title(title, color="white", fontsize=13)
    ax.axis("off")
 
    plt.tight_layout()
    plt.savefig(f"/mnt/user-data/outputs/{fname}",
                dpi=150, bbox_inches="tight", facecolor=BG)
    plt.show()
 
 
# 2-D phase diagram
 
def plot_phase_diagram(K_vals, p_vals, R):
    """
    Heatmap of steady-state order parameter r as a function of (p, K).
    """
    fig, ax = plt.subplots(figsize=(10, 7), facecolor=BG)
    ax.set_facecolor(BG)
 
    # meshgrid for pcolormesh
    im = ax.pcolormesh(np.log10(p_vals), K_vals, R,
                       cmap=CMAP_HEAT, vmin=0, vmax=1,
                       shading="auto")
 
    cbar = plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cbar.set_label("Steady-state  ⟨r⟩", color="white", fontsize=11)
    cbar.ax.yaxis.set_tick_params(color="white")
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")
 
    # Contour line at r=0.5 (rough synchronisation boundary)
    cs = ax.contour(np.log10(p_vals), K_vals, R,
                    levels=[0.5], colors=[ACCENT1], linewidths=2)
    ax.clabel(cs, fmt="r=0.5", colors=ACCENT1, fontsize=9)
 
    ax.set_xlabel("log₁₀(rewiring probability  p)", color="white", fontsize=12)
    ax.set_ylabel("Coupling strength  K", color="white", fontsize=12)
    ax.set_title("Phase diagram of synchronisation  (N=80, k=6)",
                 color="white", fontsize=13)
    ax.tick_params(colors="white")
    ax.spines[:].set_color("#444")
 
    plt.tight_layout()
    plt.savefig("/mnt/user-data/outputs/fig6_phase_diagram.png",
                dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print("✓  fig6_phase_diagram.png")
 
 
# Space-time plot (raster-style)
 
def plot_space_time(theta, t, title="Space–time phase plot", fname="fig7_spacetime.png"):
    """
    Classic space-time plot showing oscillator phases over time.
    Rows = oscillators, columns = time.  Colour = phase mod 2π.
    Visually reveals synchronisation 'waves', clusters, and phase-locking.
    """
    fig, ax = plt.subplots(figsize=(14, 7), facecolor=BG)
    ax.set_facecolor(BG)
 
    # theta shape: (time, N) → we want (N, time) for imshow
    phase_mod = np.mod(theta, 2 * np.pi).T        # (N, time)
 
    im = ax.imshow(phase_mod, aspect="auto",
                   origin="lower",
                   cmap="hsv",
                   vmin=0, vmax=2 * np.pi,
                   extent=[t[0], t[-1], 0, phase_mod.shape[0]])
 
    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label("Phase θ  [rad]", color="white")
    cbar.ax.yaxis.set_tick_params(color="white")
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")
 
    ax.set_xlabel("Time", color="white", fontsize=12)
    ax.set_ylabel("Oscillator index  i", color="white", fontsize=12)
    ax.set_title(title, color="white", fontsize=13)
    ax.tick_params(colors="white")
    ax.spines[:].set_color("#444")
 
    plt.tight_layout()
    plt.savefig(f"/mnt/user-data/outputs/{fname}",
                dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"✓  {fname}")
 
 
# Pairwise coherence matrix
 
def plot_coherence_matrix(theta_snapshot, G, A, fname="fig8_coherence.png"):
    """
    Visualise the pairwise phase-coherence matrix C[i,j] = |cos(θᵢ−θⱼ)|.
    Nodes are reordered by community (using spectral clustering on A) so
    synchronised clusters appear as bright blocks along the diagonal.
    Also overlays the adjacency matrix outline.
    """
    from scipy.linalg import eigh
 
    N = len(theta_snapshot)
    C = phase_coherence_matrix(theta_snapshot)   # (N, N)
 
    # ─ spectral reordering: use 2nd Fiedler eigenvector of Laplacian ─
    deg = A.sum(axis=1)
    D   = np.diag(deg)
    L   = D - A                    # graph Laplacian
    # compute second-smallest eigenvector
    vals, vecs = eigh(L)
    fiedler    = vecs[:, 1]        # Fiedler vector
    order      = np.argsort(fiedler)   # sort nodes by Fiedler value
 
    C_ordered = C[np.ix_(order, order)]
    A_ordered = A[np.ix_(order, order)]
 
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor=BG)
 
    for ax, mat, title, cmap in zip(
        axes,
        [C_ordered, A_ordered],
        ["Phase coherence |cos(θᵢ−θⱼ)| (Fiedler-ordered)",
         "Adjacency matrix (same ordering)"],
        [CMAP_HEAT, "Blues"]
    ):
        ax.set_facecolor(BG)
        im = ax.imshow(mat, cmap=cmap, origin="upper",
                       vmin=0, vmax=1, aspect="auto")
        cbar = plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
        cbar.ax.yaxis.set_tick_params(color="white")
        plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")
        ax.set_title(title, color="white", fontsize=11)
        ax.set_xlabel("Node (reordered)", color="white")
        ax.set_ylabel("Node (reordered)", color="white")
        ax.tick_params(colors="white")
        ax.spines[:].set_color("#444")
 
    plt.suptitle("Synchronisation structure  (spectral node ordering)",
                 color="white", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(f"/mnt/user-data/outputs/{fname}",
                dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"✓  {fname}")
 
 
# Kuramoto order-parameter trajectory on complex plane
 
def plot_complex_plane(theta, t, fname="fig9_complex_plane.png"):
    """
    Trace the order-parameter vector Z(t) = r(t) e^{iΨ(t)} in the complex plane.
    Early transient in one colour, steady state in another.
    Like a comet leaving a trail – visually captures the convergence to a
    fixed point (sync) or limit cycle (partial sync).
    """
    z   = np.mean(np.exp(1j * theta), axis=1)   # (T,) complex
    cut = int(len(t) * 0.35)                     # transient / steady split
 
    fig, ax = plt.subplots(figsize=(7, 7), facecolor=BG)
    ax.set_facecolor(BG)
 
    # Draw unit circle
    circle_theta = np.linspace(0, 2 * np.pi, 300)
    ax.plot(np.cos(circle_theta), np.sin(circle_theta),
            color="#555", lw=1, ls="--")
 
    # Transient portion – faded pink
    ax.plot(z[:cut].real, z[:cut].imag,
            color=ACCENT2, alpha=0.4, lw=0.8, label="Transient")
 
    # Steady-state portion – cyan gradient
    steady_z = z[cut:]
    for i in range(len(steady_z) - 1):
        frac = i / len(steady_z)
        col  = plt.cm.cool(frac)
        ax.plot(steady_z[i:i+2].real, steady_z[i:i+2].imag,
                color=col, lw=1.2, alpha=0.7)
 
    # Mark start and end
    ax.plot(z[0].real,  z[0].imag,  "o", color=ACCENT3, ms=10, label="Start")
    ax.plot(z[-1].real, z[-1].imag, "*", color=ACCENT1, ms=14, label="End")
 
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_aspect("equal")
    ax.set_xlabel("Re[Z]", color="white")
    ax.set_ylabel("Im[Z]", color="white")
    ax.set_title("Order-parameter  Z(t)  in complex plane", color="white", fontsize=13)
    ax.axhline(0, color="#444", lw=0.5)
    ax.axvline(0, color="#444", lw=0.5)
    ax.legend(framealpha=0.15, labelcolor="white")
    ax.tick_params(colors="white")
    ax.spines[:].set_color("#444")
    ax.text(0.7, 0.95, "|Z|=1 (full sync)", color="#888", fontsize=8,
            transform=ax.transAxes)
 
    plt.tight_layout()
    plt.savefig(f"/mnt/user-data/outputs/{fname}",
                dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"✓  {fname}")
 
 
# Local synchrony on the network layout
 
def plot_local_sync(theta, G, A, t_indices=(5, 25, -1),
                    t=None, fname="fig10_local_sync.png"):
    """
    Show the *local* order parameter rᵢ for each node (how sync is its
    neighbourhood?) at three time snapshots.
    Node colour = rᵢ, node size = degree.
    Reveals where synchrony nucleates and spreads across the network.
    """
    local_r = local_order_parameter(theta, A)   # (T, N)
 
    fig, axes = plt.subplots(1, 3, figsize=(20, 6), facecolor=BG)
    fig.suptitle("Local synchrony  rᵢ(t)  spreading on network",
                 color="white", fontsize=13, y=1.01)
 
    pos    = nx.spring_layout(G, seed=42)
    degrees = np.array([d for _, d in G.degree()])
 
    for ax, tidx in zip(axes, t_indices):
        ax.set_facecolor(BG)
        r_local = local_r[tidx]                 # (N,)
        time_label = f"t={t[tidx]:.1f}" if t is not None else f"step {tidx}"
 
        nx.draw_networkx_edges(G, pos, ax=ax,
                               alpha=0.15, edge_color="#334", width=0.6)
        sc = nx.draw_networkx_nodes(G, pos, ax=ax,
                                    node_color=r_local,
                                    cmap=CMAP_HEAT,
                                    vmin=0, vmax=1,
                                    node_size=degrees * 20 + 30)
        ax.set_title(time_label, color="white", fontsize=11)
        ax.axis("off")
 
    # single colorbar for all subplots
    sm = plt.cm.ScalarMappable(cmap=CMAP_HEAT, norm=plt.Normalize(0, 1))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes, fraction=0.02, pad=0.02)
    cbar.set_label("Local order parameter  rᵢ", color="white")
    cbar.ax.yaxis.set_tick_params(color="white")
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")
 
    plt.tight_layout()
    plt.savefig(f"/mnt/user-data/outputs/{fname}",
                dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"✓  {fname}")
 
 
# Network + clustering + L vs p comparison
 
def plot_ws_properties_vs_p(N=200, k=6,
                             p_vals=None,
                             fname="fig12_ws_properties.png"):
    """
    Reproduce the classical Watts–Strogatz figure:
    C(p)/C(0) and L(p)/L(0) vs rewiring probability.
    Overlaid with the synchronisation order parameter from sweep_p results.
    """
    if p_vals is None:
        p_vals = np.logspace(-4, 0, 30)
 
    C_list, L_list = [], []
    for p in p_vals:
        G, _ = make_ws_network(N, k, p)
        stats = network_stats(G)
        C_list.append(stats["clustering"])
        L_list.append(stats["mean_path"])
 
    C_arr = np.array(C_list)
    L_arr = np.array(L_list)
    C0    = C_arr[0]
    L0    = L_arr[0]
 
    fig, ax = plt.subplots(figsize=(9, 5), facecolor=BG)
    ax.set_facecolor(BG)
 
    ax.semilogx(p_vals, C_arr / C0, "s-", color=ACCENT1, lw=2, ms=5,
                label="C(p) / C(0)  [clustering]")
    ax.semilogx(p_vals, L_arr / L0, "o-", color=ACCENT2, lw=2, ms=5,
                label="L(p) / L(0)  [mean path length]")
 
    ax.axhspan(0, 0.05, alpha=0.08, color="cyan")
    ax.set_xlabel("Rewiring probability  p", color="white", fontsize=12)
    ax.set_ylabel("Normalised value", color="white", fontsize=12)
    ax.set_title("Watts–Strogatz properties vs rewiring probability  (N=200, k=6)",
                 color="white", fontsize=13)
    ax.legend(framealpha=0.15, labelcolor="white")
    ax.tick_params(colors="white")
    ax.spines[:].set_color("#444")
    ax.set_ylim(-0.05, 1.1)
 
    # Annotate small-world regime
    ax.axvspan(1e-3, 1e-1, alpha=0.07, color="green")
    ax.text(5e-3, 0.5, "Small-world\nregime", color="limegreen",
            fontsize=9, va="center")
 
    plt.tight_layout()
    plt.savefig(f"/mnt/user-data/outputs/{fname}",
                dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"✓  {fname}")
 
 
# Power spectrum of order parameter
 
def plot_order_param_spectrum(theta, t, fname="fig13_spectrum.png"):
    """
    Fourier power spectrum of r(t).  In the incoherent state r(t) fluctuates
    rapidly.  Near the synchronisation transition one can observe characteristic
    frequencies.  In the fully synchronised state r(t) → const → flat spectrum.
    """
    r, psi = order_parameter(theta)
 
    dt   = t[1] - t[0]
    # Use only steady-state part
    cut  = int(len(r) * 0.4)
    r_ss = r[cut:]
    psi_ss = psi[cut:]
 
    freqs_r, Pxx_r     = welch(r_ss,   fs=1/dt, nperseg=256)
    freqs_psi, Pxx_psi = welch(psi_ss, fs=1/dt, nperseg=256)
 
    fig, axes = plt.subplots(2, 1, figsize=(10, 7), facecolor=BG,
                             sharex=False)
 
    for ax, freqs, Pxx, label, col in zip(
        axes,
        [freqs_r, freqs_psi],
        [Pxx_r,   Pxx_psi],
        ["r(t)  [order parameter magnitude]",
         "Ψ(t)  [mean phase]"],
        [ACCENT1, ACCENT2]
    ):
        ax.set_facecolor(BG)
        ax.semilogy(freqs, Pxx, color=col, lw=1.5)
        ax.set_xlabel("Frequency  [1/time]", color="white")
        ax.set_ylabel("Power spectral density", color="white")
        ax.set_title(f"Power spectrum of {label}", color="white")
        ax.tick_params(colors="white")
        ax.spines[:].set_color("#444")
 
    plt.tight_layout()
    plt.savefig(f"/mnt/user-data/outputs/{fname}",
                dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"✓  {fname}")
 
 
# Synchronisation 'fingerprint' polar rose
 
def plot_polar_rose(theta_snapshots, t_indices, t,
                    fname="fig14_polar_rose.png"):
    """
    Each oscillator is plotted as a point on the unit circle (polar plot)
    at multiple time snapshots arranged in one figure.
    Colour-coded by natural frequency.
    Looks like a rose unfolding as oscillators lock together.
    """
    n_snaps = len(t_indices)
    fig, axes = plt.subplots(1, n_snaps, figsize=(5 * n_snaps, 5),
                             subplot_kw={"projection": "polar"},
                             facecolor=BG)
    if n_snaps == 1:
        axes = [axes]
 
    N      = theta_snapshots[0].shape[0]
    # dummy natural-freq ordering for colour (re-use seed 7 from earlier)
    np.random.seed(7)
    omega_dummy = np.random.normal(0, 1, N)
    freq_norm   = (omega_dummy - omega_dummy.min()) / omega_dummy.ptp()
 
    for ax, tidx, snap in zip(axes, t_indices, theta_snapshots):
        ax.set_facecolor(BG)
        ax.tick_params(colors="white")
 
        cols = plt.cm.plasma(freq_norm)
        for i, (phase, col) in enumerate(zip(snap, cols)):
            ax.plot([0, phase % (2*np.pi)], [0, 1],
                    color=col, alpha=0.4, lw=0.8)
            ax.plot(phase % (2*np.pi), 1.0,
                    "o", color=col, ms=5, alpha=0.8)
 
        # draw order parameter vector
        z   = np.mean(np.exp(1j * snap))
        r   = np.abs(z)
        psi = np.angle(z) % (2*np.pi)
        ax.annotate("", xy=(psi, r), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="->",
                                   color=ACCENT3, lw=2.5))
 
        time_label = f"t={t[tidx]:.1f}" if t is not None else f"idx {tidx}"
        ax.set_title(f"{time_label}\nr={r:.2f}", color="white",
                     fontsize=11, pad=12)
        ax.spines["polar"].set_color("#444")
        ax.set_yticklabels([])
 
    plt.suptitle("Phase distribution on unit circle  (gold arrow = order parameter)",
                 color="white", fontsize=12, y=1.03)
    plt.tight_layout()
    plt.savefig(f"/mnt/user-data/outputs/{fname}",
                dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"✓  {fname}")